<a href="https://colab.research.google.com/github/dpratap17/DeepLearningLab/blob/main/DL_LAB_EXP_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
DATA_PATH = "/content/spa.txt"

pairs = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        eng, spa = line.strip().split("\t")
        pairs.append((eng.lower(), spa.lower()))

print("Total sentence pairs:", len(pairs))

Total sentence pairs: 118121


In [5]:
def tokenize(sentence):
    return sentence.split()

special_tokens = ["<pad>", "<sos>", "<eos>", "<unk>"]

def build_vocab(sentences, min_freq=2):
    counter = Counter()
    for s in sentences:
        counter.update(tokenize(s))

    vocab = {tok: i for i, tok in enumerate(special_tokens)}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

In [6]:
src_sentences = [p[0] for p in pairs]
tgt_sentences = [p[1] for p in pairs]

SRC_VOCAB = build_vocab(src_sentences)
TGT_VOCAB = build_vocab(tgt_sentences)

SRC_IVOCAB = {i:w for w,i in SRC_VOCAB.items()}
TGT_IVOCAB = {i:w for w,i in TGT_VOCAB.items()}

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab, max_len=50):
        self.data = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def encode(self, sentence, vocab):
        tokens = tokenize(sentence)
        ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
        ids = [vocab["<sos>"]] + ids + [vocab["<eos>"]]
        ids = ids[:self.max_len]
        ids += [vocab["<pad>"]] * (self.max_len - len(ids))
        return ids

    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        return (
            torch.tensor(self.encode(src, self.src_vocab)),
            torch.tensor(self.encode(tgt, self.tgt_vocab))
        )

    def __len__(self):
        return len(self.data)

In [8]:
train_size = int(0.8 * len(pairs))
val_size = int(0.1 * len(pairs))

train_data = pairs[:train_size]
val_data = pairs[train_size:train_size+val_size]
test_data = pairs[train_size+val_size:]

train_ds = TranslationDataset(train_data, SRC_VOCAB, TGT_VOCAB)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)

        q = self.q(q).view(B, -1, self.heads, self.d_k).transpose(1,2)
        k = self.k(k).view(B, -1, self.heads, self.d_k).transpose(1,2)
        v = self.v(v).view(B, -1, self.heads, self.d_k).transpose(1,2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)

        out = out.transpose(1,2).contiguous().view(B, -1, self.heads * self.d_k)
        return self.out(out)

In [11]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        x = self.norm1(x + self.attn(x, x, x, mask))
        x = self.norm2(x + self.ff(x))
        return x


In [12]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads)
        self.cross_attn = MultiHeadAttention(d_model, heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.self_attn(x, x, x, tgt_mask))
        x = self.norm2(x + self.cross_attn(x, enc_out, enc_out, src_mask))
        x = self.norm3(x + self.ff(x))
        return x

In [13]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, heads=8, layers=4):
        super().__init__()
        self.src_emb = nn.Embedding(len(src_vocab), d_model)
        self.tgt_emb = nn.Embedding(len(tgt_vocab), d_model)
        self.pos = PositionalEncoding(d_model)

        self.encoder = nn.ModuleList([EncoderLayer(d_model, heads, 512) for _ in range(layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, heads, 512) for _ in range(layers)])

        self.fc = nn.Linear(d_model, len(tgt_vocab))

    def forward(self, src, tgt, src_mask, tgt_mask):
        src = self.pos(self.src_emb(src))
        tgt = self.pos(self.tgt_emb(tgt))

        for layer in self.encoder:
            src = layer(src, src_mask)

        for layer in self.decoder:
            tgt = layer(tgt, src, src_mask, tgt_mask)

        return self.fc(tgt)

In [14]:
model = Transformer(SRC_VOCAB, TGT_VOCAB).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=TGT_VOCAB["<pad>"])
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [15]:
def train_epoch():
    model.train()
    total_loss = 0

    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()

        output = model(src, tgt[:, :-1], None, None)
        loss = criterion(output.reshape(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [16]:
for epoch in range(20):
    loss = train_epoch()
    print(f"Epoch {epoch+1} | Loss: {loss:.4f}")

Epoch 1 | Loss: 3.0483
Epoch 2 | Loss: 1.0436
Epoch 3 | Loss: 0.4940
Epoch 4 | Loss: 0.2380
Epoch 5 | Loss: 0.1091
Epoch 6 | Loss: 0.0440
Epoch 7 | Loss: 0.0180
Epoch 8 | Loss: 0.0085
Epoch 9 | Loss: 0.0054
Epoch 10 | Loss: 0.0042


KeyboardInterrupt: 

In [17]:
from nltk.translate.bleu_score import corpus_bleu
def evaluate_bleu():
    model.eval()
    refs, hyps = [], []

    with torch.no_grad():
        for src, tgt in train_loader:
            src = src.to(device)
            output = model(src, tgt[:, :-1].to(device), None, None)
            pred = output.argmax(-1)

            for i in range(len(pred)):
                refs.append([tgt[i].tolist()])
                hyps.append(pred[i].tolist())

    return corpus_bleu(refs, hyps)
print("BLEU Score:", evaluate_bleu())

BLEU Score: 0.0934319568615016
